# Que hicimos?


## Elección de InAMP

Decidí elegir tomar la salida diferencial del sensor hacia un amplificador de instrumentación. La cuestión es si queremos alimentarlo con fuente partida o fuente simple y es ahí cuando encontramos los amplificadores **Zerø Drift** que reducen el error por offset al tener un valor de tensión muy bajo para esta especificación. Conviene ir por acá porque al detectar 0.1 Oe de mínima tendríamos una variación de unos pocos $\mu V$ a la salida del sensor, así que necesitamos que está señal no se "pierda" en el offset.

Nos decantamos por el **INA333** de *Texas Instruments*, con el objetivo de simplificar el diseño al colocar fuente simple. Eso sí, debemos colocar un valor de referencia en el pin 4 de $V-$ para que, en caso de tener valores negativos de campo, la salida del amplificador no sature. Elijo poner 1,65V en este pin convenientemente por que se trata de la mitad del rango de tensión de mi conversor ADC (de 0 a 3,3V).

**OJO:** La entrada de este pin es de baja impedancia, por lo que no puedo colocar un simple divisor resistivo porque arruinaria la relación de rechazo de Modo Común del InAMP, en su lugar debería colocar un buffer a la salida del divisor.

La ganancia del InAMP queda definida por:

$$G = 1 + \frac{100 K \Omega}{R_G}$$

Debemos asegurarnos de que no se supere el rango de los 3,3V, para eso vamos a definir un valor de $R_G$ que nos permita no superar ese rango. Vamos a utilizar el dato de la sensibilidad del sensor el cual varía desde $0.02 \frac{mV}{V \dot \ Oe}$ a $0.03 \frac{mV}{V \dot \ Oe}$; esto se lee como: 

$$\frac{milivolt \ de \ salida}{volt \ de \ alimentación \ X \ campo \ aplicado}$$

Para sacar la tensión maxima de salida del sensor vamos a usar el peor caso tanto de sensibilidad como de campo aplicado (que es de 175 Oe).

$$\Delta V_{IN} = sensibilidad \ \ V_{alimentación} \  \ H_{max} \rightarrow 0.02 \frac{mV}{V \ Oe} \ 5 V \ 175 Oe = 17.5 mV$$

Esta es la tensión máxima que ingresará al InAMP, con ese dato sacamos la ganancia máxima que se puede obtener del InAMP, que es la que la nos hace obtener $1.65V$ a la salida.


$$G = \frac{1.65 \ V}{17.5 \ mV} \approx 94.28$$


Y sabiendo esto podemos sacar el valor de $R_G$ que cumpla:

$$R_G = \frac{100 K \Omega}{ G - 1} \rightarrow  \frac{100 K \Omega}{ 62.85 - 1} \approx 1.072 K \Omega$$

Elijo $R_G = 1 K \Omega $ 

## Justificación de bits de ADC

Necesitamos una resolución de bits capaz de detectar hasta $0.1 \ Oe$. Primero vamos a calcular la tensión que ingresa al InAMP:

$$0.02 \frac{mV}{V \ Oe} \ 5 V \ 0.1 Oe = 10 \mu V$$

Estos $10 \ \mu V$ los pasamos a la salida multiplicando por la ganancia del InAMP

$$V_{out} =  94.28 \ 10 \mu V = 0.943mV $$

$0.943 \ mV$ es lo mínimo que debería leer el ADC, osea el bit menos significativo del ADC debe ser menor a este número. Sabiendo que la escala del ADC es de $0 \ V$ a $3.3 \ V$ podemos decir que:

$$\frac{3.3V}{2^{N}} \leq 0.943mV  \rightarrow N \geq log_{2} (\frac{3.3V}{0.943 mV}) \rightarrow N \geq 11.77 \ bits \approx 12 \ bits$$

Matemáticamente llegamos a que la resolución mínima de nuestro ADC debe ser de 12 bits para poder leer hasta $0.1 \ Oe$ de campo.